# Profile PyHEARTS 2.0

Measure end-to-end runtime and separate the validated 2025 core from the record-level STPQ T post-pass.

Install optional simulation support first: `python -m pip install -e ".[sim]"`.

In [ ]:
from __future__ import annotations

import cProfile
import io
import pstats
import time

import neurokit2 as nk
import numpy as np
import pandas as pd

import pyhearts
from pyhearts import PyHEARTS
import pyhearts.core.hybrid as hybrid_mod

FS = 500.0
DURATION_S = 30
ECG = np.asarray(
    nk.ecg_simulate(
        duration=DURATION_S,
        sampling_rate=int(FS),
        heart_rate=70,
        random_state=42,
    ),
    dtype=float,
)

print("package:", pyhearts.__version__, pyhearts.__file__)
print(f"signal: {len(ECG)} samples ({DURATION_S} s @ {FS:g} Hz)")

## End-to-end runtime

Preprocessing is measured separately from analysis so device-specific filtering choices remain visible.

In [ ]:
analyzer = PyHEARTS(sampling_rate=FS, species="human")

t0 = time.perf_counter()
filtered = analyzer.preprocess_signal(
    ECG,
    highpass_cutoff=0.5,
    lowpass_cutoff=50.0,
    filter_order=4,
    notch_frequency=50.0,
    quality_factor=30.0,
)
preprocess_s = time.perf_counter() - t0

t0 = time.perf_counter()
features, cycles = analyzer.analyze_ecg(filtered)
analyze_s = time.perf_counter() - t0

pd.DataFrame(
    [
        {"stage": "preprocess", "seconds": preprocess_s},
        {"stage": "analyze", "seconds": analyze_s},
        {"stage": "total", "seconds": preprocess_s + analyze_s},
    ]
).assign(realtime_multiple=lambda x: DURATION_S / x["seconds"])

print(f"cycles={len(features)}; finite T={features['T_global_center_idx'].notna().sum()}")

## Core versus STPQ timing

This wraps the two production stages used by the public hybrid analyzer.

In [ ]:
analyzer = PyHEARTS(sampling_rate=FS, species="human")
stage_times = {}

original_core = analyzer._core.analyze_ecg
original_stpq = hybrid_mod.detect_record_stpq_t

def timed_core(*args, **kwargs):
    t0 = time.perf_counter()
    try:
        return original_core(*args, **kwargs)
    finally:
        stage_times["2025 core"] = time.perf_counter() - t0

def timed_stpq(*args, **kwargs):
    t0 = time.perf_counter()
    try:
        return original_stpq(*args, **kwargs)
    finally:
        stage_times["record STPQ T"] = time.perf_counter() - t0

analyzer._core.analyze_ecg = timed_core
hybrid_mod.detect_record_stpq_t = timed_stpq
try:
    t0 = time.perf_counter()
    features, cycles = analyzer.analyze_ecg(filtered)
    stage_times["hybrid total"] = time.perf_counter() - t0
finally:
    analyzer._core.analyze_ecg = original_core
    hybrid_mod.detect_record_stpq_t = original_stpq

pd.DataFrame(
    [{"stage": name, "seconds": seconds} for name, seconds in stage_times.items()]
).sort_values("seconds", ascending=False)

## Python hotspots

`cProfile` identifies the functions with the largest cumulative runtime.

In [ ]:
profiler = cProfile.Profile()
analyzer = PyHEARTS(sampling_rate=FS, species="human")
profiler.enable()
analyzer.analyze_ecg(filtered)
profiler.disable()

stream = io.StringIO()
pstats.Stats(profiler, stream=stream).strip_dirs().sort_stats("cumulative").print_stats(30)
print(stream.getvalue())